In [19]:
import os
from ollama import chat
import json
import requests
import re


In [20]:
def analisar_comentarios_geral():
    try:
    
        with open("comentarios.json", "r", encoding="utf-8") as f:
            comentarios = json.load(f)

        comentarios_texto = "\n".join(f"- {c}" for c in comentarios)

        prompt = f"""
Você é um especialista em análise de feedback educacional.

Analise os comentários abaixo e gere uma visão geral.

Gere também um score de 0 a 10:
- 0 = totalmente negativo
- 5 = neutro
- 10 = totalmente positivo

Responda APENAS em JSON válido:
{{
  "sentimento_geral": "positivo | negativo | neutro",
  "score": 0,
  "resumo": "resumo geral dos comentários com no maximo 100 caracteres"
}}

Comentários:
{comentarios_texto}
"""

        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3",
                "prompt": prompt,
                "stream": False
            }
        )

        data = response.json()

        if "response" not in data:
            return {
                "erro": "Resposta inesperada da API",
                "detalhes": data
            }

        texto = data["response"].strip()

        texto = texto.replace("```json", "").replace("```", "").strip()

        match = re.search(r"\{.*\}", texto, re.DOTALL)

        if not match:
            return {
                "erro": "Não foi possível extrair JSON",
                "resposta": texto
            }

        texto_json = match.group(0)

        try:
            return json.loads(texto_json)
        except json.JSONDecodeError:
            return {
                "erro": "JSON inválido mesmo após extração",
                "resposta": texto_json
            }

    except Exception as e:
        return {
            "erro": "Falha na execução",
            "detalhes": str(e)
        }

In [21]:
resultado = analisar_comentarios_geral()

import json
print(json.dumps(resultado, indent=2, ensure_ascii=False))

{
  "sentimento_geral": "positivo",
  "score": 8,
  "resumo": "O curso foi bem recebido com destaque para o conteúdo interessante, exemplos práticos e entrevistas com especialistas. Algumas críticas foram feitas sobre a velocidade do curso e a falta de recursos adicionais, mas em geral, os participantes foram satisfeitos."
}
